# Emotion Classifier API Server (Ngrok)
This notebook mounts your Google Drive, loads your trained emotion classification model, and exposes it securely to the public internet using **FastAPI** and **Ngrok** so your backend can communicate with it.

In [1]:
# 1. Install Required Libraries
!pip install fastapi uvicorn pyngrok nest-asyncio joblib scikit-learn transformers

In [2]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
from transformers import pipeline

# Direct path to your pre-unzipped model folder
MODEL_PATH = "/content/drive/MyDrive/models/roberta_emotion_model"

# Verify the folder exists before loading to catch potential path typos
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Could not find the unzipped model directory at '{MODEL_PATH}'. "
        "Please double-check the path name inside your Google Drive."
    )

print(f"Loading RoBERTa model directly from local directory: {MODEL_PATH}...")

# Load the classifier directly from the local folder files
classifier = pipeline(
    "text-classification",
    model=MODEL_PATH,
    tokenizer=MODEL_PATH
)

print("Model loaded successfully from local folder!")

Loading RoBERTa model directly from local directory: /content/drive/MyDrive/models/roberta_emotion_model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully from local folder!


In [4]:

# 4. Define FastAPI Server
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Emotion Classifier API")

class EmotionRequest(BaseModel):
    text: str

@app.get("/")
def read_root():
    return {"status": "Emotion Classifier is running!"}

@app.post("/predict")
def predict_emotion(request: EmotionRequest):
    try:
        # Use HuggingFace pipeline for prediction
        result = classifier(request.text)[0]
        emotion = result["label"]
        score = result["score"]

        return {"emotion": str(emotion), "score": float(score)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [5]:
# 5. Start Ngrok & Uvicorn
import nest_asyncio
import uvicorn
import os
import asyncio
from pyngrok import ngrok
from google.colab import userdata

# Authenticate Ngrok (Add NGROK_AUTHTOKEN to your Colab Secrets!)
ngrok_token = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_token:
    print("WARNING: NGROK_AUTHTOKEN not found in secrets. Ngrok might time out or restrict your session.")
else:
    ngrok.set_auth_token(ngrok_token)

# Kill any existing tunnels to avoid conflicts
ngrok.kill()

# Open an ngrok tunnel to the dev server (port 8000)
ngrok_tunnel = ngrok.connect(8000)
public_url = ngrok_tunnel.public_url

print("=" * 60)
print("\n\n🚀 YOUR PUBLIC API URL IS ALIVE 🚀")
print(f"URL: {public_url}\n")

# Apply nest_asyncio to allow Uvicorn to run inside the Colab Jupyter loop
nest_asyncio.apply()

# Start the server using a config and manual run to avoid the asyncio.run() error
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Run the server in the existing event loop
await server.serve()



🚀 YOUR PUBLIC API URL IS ALIVE 🚀
URL: https://abiding-reoccur-humid.ngrok-free.dev



INFO:     Started server process [8093]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Ngrok URL successfully saved to Google Drive: /content/drive/MyDrive/models/deploy/url.txt
Your local backend can now dynamically read this file!

INFO:     197.55.27.83:0 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [8093]
